#### The game plan is as follows.
We'll start with two chains, one suggesting three intermediate level programming language books, and
another suggesting three projects on the same programming language at the same level.
Both chains will accept one input, namely the language.
Contrary to previous discussions, however, our purpose won't be connecting them to form a longer chain,
but instead running them simultaneously.
Finally, we'll study the graph and execution time of a runnable parallel object, demonstrating a significant
increase in speed compared to invoking the chains sequentially With the strategy laid out.
Let's get to the implementation.

In [1]:
import config
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

C:\Users\moham\anaconda3\envs\Langchain_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
chat_template_books = ChatPromptTemplate.from_template(''' 
Suggest three of the best intermediate level {programming language} books.
Answer only by listing the books.
''')

chat_template_projects = ChatPromptTemplate.from_template(''' 
Suggest three interesting {programming language} projects suitable for intermediate level programmers.
Answer only by listing the projects.
''')

In [3]:
chat = ChatOpenAI(
    model = "gpt-3.5-turbo",
    temperature = 0,
    seed = 365,
    max_tokens = 300,
    openai_api_key = config.api_key
)

In [4]:
string_parser =  StrOutputParser()

In [5]:
chain_books = chat_template_books | chat | string_parser
chain_projects = chat_template_projects | chat | string_parser

In [ ]:
#Create a new variable called chain parallel and set it equal to the named class as an argument.
#Pass a dictionary with two key value pairs.
#You can set any keys you like.

chain_parallel = RunnableParallel({'books': chain_books,'projects': chain_projects})

In [ ]:
chain_parallel.invoke({'programming language': 'python'})

In [ ]:
chain_parallel.get_graph().print_ascii() #library that allowed for visualizing chain nodes and their sequence through an Ascii print.

#### It's worth noting the similarities and differences between the batch method we encountered earlier and
#### the runnable parallel class.
#### The batch method allowed us to invoke the same runnable with different input values.
#### In contrast, we can execute several Runnables using identical input values with the runnable parallel
#### class.

In [ ]:
%%time 
chain_books.invoke({'programming language': 'python'})

In [ ]:
%%time 
chain_projects.invoke({'programming language': 'python'})

In [ ]:
chain_parallel.invoke({'programming language': 'python'})

We can conclusively state that invoking runnables in parallel is more time efficient than invoking them.
Consequentially.


## NEXT
#### we'll pipe a runnable parallel object with another runnable parallel object into a chain

We'll now extend this vital discussion on runnable parallel objects by passing the books and projects
from the last lesson into a new prompt template, asking for an estimate of the time it would take to
complete them.


In [6]:
chat_template_time = ChatPromptTemplate.from_template(
     '''
     I'm an intermediate level programmer.
     
     Consider the following literature:
     {books}
     
     Also, consider the following projects:
     {projects}
     
     Roughly how much time would it take me to complete the literature and the projects?
     
     '''
)

In [8]:
chain_time = ( RunnableParallel({'books': chain_books,
                                                 'projects': chain_projects})
                                                | chat_template_time 
                                                | chat | string_parser
)    

chain_time2 = ({'books': chain_books,
                'projects': chain_projects}
                | chat_template_time 
                | chat | string_parser                              
)    

#When we pipe together a runnable parallel object with another runnable.
#To construct a chain, we don't need to wrap the dictionary in a runnable parallel class.
#This is done automatically.

In [9]:
print(chain_time.invoke({'programming language': 'python'}))

Completing the literature mentioned would depend on your reading speed and comprehension level. "Fluent Python" is around 792 pages, "Python Cookbook" is around 706 pages, and "Effective Python" is around 256 pages. If you dedicate a few hours each day to reading, it could take you a few weeks to a couple of months to finish all three books.

As for the projects, the time required would vary based on your familiarity with the technologies involved and the complexity of the tasks. Building a web scraper could take a few days to a couple of weeks, creating a chatbot might take a few weeks to a month, and developing a machine learning model could take a few weeks to a few months depending on the amount of data and the complexity of the model.

Overall, if you dedicate consistent time and effort to both the literature and the projects, you could potentially complete everything within a few months to a year. It's important to break down the tasks into smaller manageable chunks and stay focu

In [10]:
chain_time.get_graph().print_ascii() #library that allowed for visualizing chain nodes and their sequence through an Ascii print.

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            